# Dashboard Qualité de Service — SNCB / Infrabel

**Auteur** : Tahar Guenfoud    
**Source** : [Open Data Infrabel](https://opendata.infrabel.be)

---

## Objectif
Construire un dashboard interactif pour analyser la ponctualité et la fiabilité du réseau ferroviaire belge.

## Plan du notebook
| Étape | Description |
|---|---|
| **1. Extract** | Téléchargement des 5 sources Open Data Infrabel |
| **2. Transform** | Nettoyage, typage, colonnes calculées |
| **3. EDA** | Analyse exploratoire — tendances, anomalies |
| **4. KPIs** | Calcul des indicateurs métier (Ponctualité, Reliability, Minutes perdues) |
| **5. Visualisation** | Graphiques pour le dashboard |

---
## 0. Imports & Configuration

In [2]:
import requests
import pandas as pd
import os
import time

# Dossier de sauvegarde
RAW_DIR   = "data/raw"
CLEAN_DIR = "data/clean"

print("✅ Imports OK")

✅ Imports OK


---
# ÉTAPE 1 — Extract
Téléchargement des 5 datasets depuis l'API Open Data Infrabel.

In [3]:
BASE = (
    "https://opendata.infrabel.be/api/explore/v2.1/catalog/datasets"
    "/{}/exports/csv?lang=fr&timezone=Europe%2FBrussels&use_labels=true&delimiter=%3B"
)

DATASETS = {
    "ponctualite_par_gare"    : "maandelijkse-stiptheid-per-stopplaats",
    "causes_retards"          : "oorzaken-vertraging-per-maand",
    "ponctualite_par_moment"  : "nationale-stiptheid-per-moment-en-per-maand",
    "trains_supprimes"        : "afgeschafte-treinen-per-maand-vanaf-2020",
    "kpi_contrat_performance" : "indicatoren-performantie-contract",
}

dfs = {}
for name, dataset_id in DATASETS.items():
    dfs[name] = pd.read_csv(BASE.format(dataset_id), sep=";")
    print(f"✅ {name:<35} {dfs[name].shape[0]:>6,} lignes × {dfs[name].shape[1]} cols")

✅ ponctualite_par_gare                27,343 lignes × 13 cols
✅ causes_retards                         425 lignes × 14 cols
✅ ponctualite_par_moment                 484 lignes × 9 cols
✅ trains_supprimes                        73 lignes × 7 cols
✅ kpi_contrat_performance                177 lignes × 13 cols


---
# ÉTAPE 2 — Transform
Nettoyage et préparation de chaque dataset.

### 2.1 — Ponctualité par Gare

In [12]:
# Explorer les colonnes brutes
df = dfs["ponctualite_par_gare"]
df.tail(10)

,Date,Point d'arrêt,Point d'arrêt.1,Point d'arrêt.2,ID point opérationnel,Classification,Classification.1,Classification.2,Ponctualité,Nombre total de trains,Nombre de trains ponctuels,Geo Point,Geo Shape
27333,2023-11,FRAMERIES,FRAMERIES,FRAMERIES,422,Stopplaats,Point d'arrêt,Stopping point,87.196468,906.0,790.0,"50.40552689016233, 3.906385857023495","{""coordinates"": [3.906385857023495, 50.4055268..."
27334,2023-11,FEXHE-LE-HAUT-CLOCHER,FEXHE-LE-HAUT-CLOCHER,FEXHE-LE-HAUT-CLOCHER,399,Stopplaats,Point d'arrêt,Stopping point,89.730290,964.0,865.0,"50.66429943269758, 5.397266518195518","{""coordinates"": [5.397266518195518, 50.6642994..."
27335,2023-11,FLEMALLE-GRANDE,FLEMALLE-GRANDE,FLEMALLE-GRANDE,401,Stopplaats,Point d'arrêt,Stopping point,91.383220,882.0,806.0,"50.605259633719655, 5.48108901393654","{""coordinates"": [5.48108901393654, 50.60525963..."
27336,2023-11,FLOREFFE,FLOREFFE,FLOREFFE,406,Stopplaats,Point d'arrêt,Stopping point,90.692124,1257.0,1140.0,"50.44348110123118, 4.762999764162857","{""coordinates"": [4.762999764162857, 50.4434811..."
27337,2023-11,FLORIVAL,FLORIVAL,FLORIVAL,410,Stopplaats,Point d'arrêt,Stopping point,94.306050,1405.0,1325.0,"50.76145683011463, 4.654204552903399","{""coordinates"": [4.654204552903399, 50.7614568..."
27338,2023-11,GONTRODE,GONTRODE,GONTRODE,474,Stopplaats,Point d'arrêt,Stopping point,79.947230,1137.0,909.0,"50.979840155563366, 3.801555019245788","{""coordinates"": [3.801555019245788, 50.9798401..."
27339,2023-11,FAUX,FAUX,FAUX,395,Stopplaats,Point d'arrêt,Stopping point,90.492170,894.0,809.0,"50.621786596713214, 4.549374449548871","{""coordinates"": [4.549374449548871, 50.6217865..."
27340,2023-11,BALEGEM-ZUID,BALEGEM-ZUID,BALEGEM-ZUID,106,Stopplaats,Point d'arrêt,Stopping point,79.525483,1138.0,905.0,"50.90080462540978, 3.805776391176711","{""coordinates"": [3.805776391176711, 50.9008046..."
27341,2023-11,AYE,AYE,AYE,100,Stopplaats,Point d'arrêt,Stopping point,95.246801,547.0,521.0,"50.22485292105391, 5.300638767374593","{""coordinates"": [5.300638767374593, 50.2248529..."
27342,2023-11,BAS-OHA,BAS-OHA,BAS-OHA,118,Stopplaats,Point d'arrêt,Stopping point,86.473430,828.0,716.0,"50.52248171944925, 5.191103060283717","{""coordinates"": [5.191103060283717, 50.5224817..."


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27343 entries, 0 to 27342
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Date                        27343 non-null  object 
 1   Point d'arrêt               27343 non-null  object 
 2   Point d'arrêt.1             27343 non-null  object 
 3   Point d'arrêt.2             27343 non-null  object 
 4   ID point opérationnel       27343 non-null  int64  
 5   Classification              27343 non-null  object 
 6   Classification.1            27343 non-null  object 
 7   Classification.2            27343 non-null  object 
 8   Ponctualité                 27343 non-null  float64
 9   Nombre total de trains      27343 non-null  float64
 10  Nombre de trains ponctuels  27343 non-null  float64
 11  Geo Point                   27307 non-null  object 
 12  Geo Shape                   27307 non-null  object 
dtypes: float64(3), int64(1), object

In [13]:
df.isnull().sum()

Date                           0
Point d'arrêt                  0
Point d'arrêt.1                0
Point d'arrêt.2                0
ID point opérationnel          0
Classification                 0
Classification.1               0
Classification.2               0
Ponctualité                    0
Nombre total de trains         0
Nombre de trains ponctuels     0
Geo Point                     36
Geo Shape                     36
dtype: int64

In [15]:
df["Date"].unique()

array(['2023-11', '2023-12', '2024-01', '2024-02', '2022-10', '2022-11',
       '2024-12', '2025-01', '2022-12', '2025-02', '2023-01', '2025-03',
       '2023-04', '2025-04', '2023-05', '2025-05', '2023-06', '2025-06',
       '2023-02', '2025-07', '2023-03', '2025-08', '2022-08', '2025-09',
       '2022-09', '2025-10', '2025-11', '2025-12', '2024-04', '2024-05',
       '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11',
       '2026-01', '2024-03', '2022-04', '2022-06', '2022-07', '2022-01',
       '2022-02', '2022-05', '2022-03', '2023-07', '2023-08', '2023-09',
       '2023-10'], dtype=object)

In [18]:
df_gare = dfs["ponctualite_par_gare"].copy()

# Renommer (13 colonnes dans l'ordre exact)
df_gare.columns = [
    "date",
    "nom_gare_fr",
    "nom_gare_nl",
    "nom_gare_de",
    "id_gare",
    "classification_fr",
    "classification_nl",
    "classification_de",
    "ponctualite_pct",
    "nb_trains",
    "nb_trains_ponctuels",
    "geo_point",
    "geo_shape"
]

# Parser la date
df_gare["date"] = pd.to_datetime(df_gare["date"], format="%Y-%m")

# Vérification
print(f"Shape : {df_gare.shape}")
print(f"Valeurs manquantes :\n{df_gare.isnull().sum()}")
df_gare.head(3)

Shape : (27343, 13)
Valeurs manquantes :
date                    0
nom_gare_fr             0
nom_gare_nl             0
nom_gare_de             0
id_gare                 0
classification_fr       0
classification_nl       0
classification_de       0
ponctualite_pct         0
nb_trains               0
nb_trains_ponctuels     0
geo_point              36
geo_shape              36
dtype: int64


,date,nom_gare_fr,nom_gare_nl,nom_gare_de,id_gare,classification_fr,classification_nl,classification_de,ponctualite_pct,nb_trains,nb_trains_ponctuels,geo_point,geo_shape
0,2023-11-01,BEIGNEE,BEIGNEE,BEIGNEE,133,Stopplaats,Point d'arrêt,Stopping point,91.200000,250.0,228.0,"50.33322928558822, 4.406633114707752","{""coordinates"": [4.406633114707752, 50.3332292..."
1,2023-11-01,BELSELE,BELSELE,BELSELE,138,Stopplaats,Point d'arrêt,Stopping point,83.060453,1588.0,1319.0,"51.15105293281537, 4.088971326009601","{""coordinates"": [4.088971326009601, 51.1510529..."
2,2023-11-01,BERLAAR,BERLAAR,BERLAAR,142,Stopplaats,Point d'arrêt,Stopping point,88.265746,1159.0,1023.0,"51.113708436411315, 4.638257976114972","{""coordinates"": [4.638257976114972, 51.1137084..."


In [19]:
# Voir quelques exemples côte à côte
df_gare[["nom_gare_fr", "nom_gare_nl", "nom_gare_de"]].drop_duplicates().head(20)

,nom_gare_fr,nom_gare_nl,nom_gare_de
0,BEIGNEE,BEIGNEE,BEIGNEE
1,BELSELE,BELSELE,BELSELE
2,BERLAAR,BERLAAR,BERLAAR
3,ANTWERPEN-LUCHTBAL,ANTWERPEN-LUCHTBAL,ANTWERPEN-LUCHTBAL
4,AARSELE,AARSELE,AARSELE
5,AISEAU,AISEAU,AISEAU
6,BAASRODE-ZUID,BAASRODE-ZUID,BAASRODE-ZUID
7,ANTWERPEN-ZUID,ANTWERPEN-ZUID,ANTWERPEN-ZUID
8,APPELTERRE,APPELTERRE,APPELTERRE
9,ARCHENNES,ARCHENNES,ARCHENNES


In [20]:
# Combien de fois les 3 colonnes sont identiques ?
identiques = (df_gare["nom_gare_fr"] == df_gare["nom_gare_nl"]).sum()
print(f"FR = NL : {identiques} fois sur {len(df_gare)}")

identiques2 = (df_gare["nom_gare_fr"] == df_gare["nom_gare_de"]).sum()
print(f"FR = DE : {identiques2} fois sur {len(df_gare)}")

FR = NL : 26046 fois sur 27343
FR = DE : 26046 fois sur 27343


In [21]:
# Supprimer les colonnes inutiles
df_gare = df_gare.drop(columns=["nom_gare_nl", "nom_gare_de",
                                 "classification_nl", "classification_de",
                                 "geo_shape"])

print(f"Colonnes restantes : {df_gare.columns.tolist()}")
print(f"Shape : {df_gare.shape}")

Colonnes restantes : ['date', 'nom_gare_fr', 'id_gare', 'classification_fr', 'ponctualite_pct', 'nb_trains', 'nb_trains_ponctuels', 'geo_point']
Shape : (27343, 8)


In [24]:
display(df_gare.head())
df_gare.info()

,date,nom_gare_fr,id_gare,classification_fr,ponctualite_pct,nb_trains,nb_trains_ponctuels,geo_point
0,2023-11-01,BEIGNEE,133,Stopplaats,91.200000,250.0,228.0,"50.33322928558822, 4.406633114707752"
1,2023-11-01,BELSELE,138,Stopplaats,83.060453,1588.0,1319.0,"51.15105293281537, 4.088971326009601"
2,2023-11-01,BERLAAR,142,Stopplaats,88.265746,1159.0,1023.0,"51.113708436411315, 4.638257976114972"
3,2023-11-01,ANTWERPEN-LUCHTBAL,764,Stopplaats,80.809077,2027.0,1638.0,"51.24356131707571, 4.424880242553035"
4,2023-11-01,AARSELE,10,Stopplaats,74.226804,97.0,72.0,"50.98449587636041, 3.41837146665051"


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27343 entries, 0 to 27342
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   date                 27343 non-null  datetime64[ns]
 1   nom_gare_fr          27343 non-null  object        
 2   id_gare              27343 non-null  int64         
 3   classification_fr    27343 non-null  object        
 4   ponctualite_pct      27343 non-null  float64       
 5   nb_trains            27343 non-null  float64       
 6   nb_trains_ponctuels  27343 non-null  float64       
 7   geo_point            27307 non-null  object        
dtypes: datetime64[ns](1), float64(3), int64(1), object(3)
memory usage: 1.7+ MB


### 2.2 — Causes des Retards

### Colonnes clés — Causes des Retards

| Colonne | Description |
|---|---|
| `responsable` | Qui a causé le retard : Infrabel / SNCB / Tiers / Robustesse systémique / Autres |
| `nb_trains_en_retard` | Nombre de trains en retard imputés à ce responsable ce mois |
| `nb_trains_total` | Nombre total de trains observés ce mois |
| `perte_ponctualite` | Minutes de retard totales causées par ce responsable ce mois |
| `proportion_pct` | Part (%) de ce responsable dans les retards du mois |
| `nb_trains_en_retard_ytd` | Cumul des trains en retard depuis le 1er janvier (YTD) |
| `nb_trains_total_ytd` | Cumul total des trains depuis le 1er janvier (YTD) |
| `perte_ponctualite_ytd` | Cumul des minutes de retard depuis le 1er janvier (YTD) |
| `proportion_ytd_pct` | Part (%) cumulée depuis le 1er janvier (YTD) |

> **YTD (Year-To-Date)** : cumul depuis le 1er janvier de l'année en cours.
> Permet de suivre la tendance annuelle indépendamment des variations mensuelles.

In [26]:
# Explorer les colonnes brutes
df = dfs["causes_retards"]
display(df.head(3))
df.info()

,Année,Année/mois,Mois,Responsable NL,Responsable,Responsable EN,Nombre de repérages de trains en retard,Nombre total de repérages de trains,Perte de ponctualité,% proportion,Nombre de repérages de trains en retard YTD,Nombre total de repérages de trains YTD,Perte de ponctualité YTD,% proportion YTD
0,2026,2026-01,1,Robuustheid van het systeem,Robustesse systémique,Systemic robustness,1440.909241,107866,1.34,17.56,1440.909241,107866,1.34,17.56
1,2026,2026-01,1,Derden,Tiers,Third parties,1748.080249,107866,1.62,21.31,1748.080249,107866,1.62,21.31
2,2026,2026-01,1,Andere,Autres,Others,271.613784,107866,0.25,3.31,271.613784,107866,0.25,3.31


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 425 entries, 0 to 424
Data columns (total 14 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   Année                                        425 non-null    int64  
 1   Année/mois                                   425 non-null    object 
 2   Mois                                         425 non-null    int64  
 3   Responsable NL                               425 non-null    object 
 4   Responsable                                  425 non-null    object 
 5   Responsable EN                               425 non-null    object 
 6   Nombre de repérages de trains en retard      425 non-null    float64
 7   Nombre total de repérages de trains          425 non-null    int64  
 8   Perte de ponctualité                         425 non-null    float64
 9   % proportion                                 425 non-null    float64
 10  No

In [27]:
df_causes = dfs["causes_retards"].copy()

print(f"Shape : {df_causes.shape}")
print(f"Valeurs manquantes :\n{df_causes.isnull().sum()}")
df_causes.head(3)

Shape : (425, 14)
Valeurs manquantes :
Année                                          0
Année/mois                                     0
Mois                                           0
Responsable NL                                 0
Responsable                                    0
Responsable EN                                 0
Nombre de repérages de trains en retard        0
Nombre total de repérages de trains            0
Perte de ponctualité                           0
% proportion                                   0
Nombre de repérages de trains en retard YTD    0
Nombre total de repérages de trains YTD        0
Perte de ponctualité YTD                       0
% proportion YTD                               0
dtype: int64


,Année,Année/mois,Mois,Responsable NL,Responsable,Responsable EN,Nombre de repérages de trains en retard,Nombre total de repérages de trains,Perte de ponctualité,% proportion,Nombre de repérages de trains en retard YTD,Nombre total de repérages de trains YTD,Perte de ponctualité YTD,% proportion YTD
0,2026,2026-01,1,Robuustheid van het systeem,Robustesse systémique,Systemic robustness,1440.909241,107866,1.34,17.56,1440.909241,107866,1.34,17.56
1,2026,2026-01,1,Derden,Tiers,Third parties,1748.080249,107866,1.62,21.31,1748.080249,107866,1.62,21.31
2,2026,2026-01,1,Andere,Autres,Others,271.613784,107866,0.25,3.31,271.613784,107866,0.25,3.31


In [28]:
# Valeurs uniques dans chaque colonne
print("FR :", df["Responsable"].unique())
print("NL :", df["Responsable NL"].unique())
print("EN :", df["Responsable EN"].unique())

FR : ['Robustesse systémique' 'Tiers' 'Autres' 'Infrabel' 'SNCB']
NL : ['Robuustheid van het systeem' 'Derden' 'Andere' 'Infrabel' 'NMBS']
EN : ['Systemic robustness' 'Third parties' 'Others' 'Infrabel' 'SNCB/NMBS']


In [29]:
# Vérifier si FR et NL sont toujours différents
identiques = (df["Responsable"] == df["Responsable NL"]).sum()
print(f"FR = NL : {identiques} fois sur {len(df)}")

identiques2 = (df["Responsable"] == df["Responsable EN"]).sum()
print(f"FR = EN : {identiques2} fois sur {len(df)}")

FR = NL : 85 fois sur 425
FR = EN : 85 fois sur 425


In [30]:
df_causes.columns = [
    "annee", "date", "mois",
    "responsable_nl", "responsable", "responsable_en",
    "nb_trains_en_retard", "nb_trains_total",
    "perte_ponctualite", "proportion_pct",
    "nb_trains_en_retard_ytd", "nb_trains_total_ytd",
    "perte_ponctualite_ytd", "proportion_ytd_pct"
]

# Parser la date
df_causes["date"] = pd.to_datetime(df_causes["date"], format="%Y-%m")

# Supprimer les colonnes redondantes
df_causes = df_causes.drop(columns=["annee", "mois", "responsable_nl", "responsable_en"])

print(f"Shape : {df_causes.shape}")
print(f"\nResponsables : {df_causes['responsable'].unique()}")
df_causes.head()

Shape : (425, 10)

Responsables : ['Robustesse systémique' 'Tiers' 'Autres' 'Infrabel' 'SNCB']


,date,responsable,nb_trains_en_retard,nb_trains_total,perte_ponctualite,proportion_pct,nb_trains_en_retard_ytd,nb_trains_total_ytd,perte_ponctualite_ytd,proportion_ytd_pct
0,2026-01-01,Robustesse systémique,1440.909241,107866,1.34,17.56,1440.909241,107866,1.34,17.56
1,2026-01-01,Tiers,1748.080249,107866,1.62,21.31,1748.080249,107866,1.62,21.31
2,2026-01-01,Autres,271.613784,107866,0.25,3.31,271.613784,107866,0.25,3.31
3,2026-01-01,Infrabel,945.015104,107866,0.88,11.52,945.015104,107866,0.88,11.52
4,2026-01-01,SNCB,3798.381622,107866,3.52,46.30,3798.381622,107866,3.52,46.30


### 2.3 — Ponctualité par Moment

In [32]:
df = dfs["ponctualite_par_moment"]
display(df.head())
df.info()

,Mois,Instant,Instant.1,Instant.2,Ponctualité,Nombre de trains,Trains ayant moins de 6 min. de retard,Minutes de retard,Année
0,2016-01,Weekends,Weekends,Weekends,93.869404,24549,23044,32326,2016
1,2016-01,Daluren,Heures creuses,Off-peak hours,90.689997,45435,41205,93564,2016
2,2016-02,Avondspits,Pointe du soir,Evening peak hour,88.282383,17239,15219,40597,2016
3,2016-03,Ochtendspits,Pointe du matin,Morning peak hour,88.131496,16548,14584,40055,2016
4,2016-03,Avondspits,Pointe du soir,Evening peak hour,88.470725,17165,15186,41783,2016


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 484 entries, 0 to 483
Data columns (total 9 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Mois                                    484 non-null    object 
 1   Instant                                 484 non-null    object 
 2   Instant.1                               484 non-null    object 
 3   Instant.2                               484 non-null    object 
 4   Ponctualité                             484 non-null    float64
 5   Nombre de trains                        484 non-null    int64  
 6   Trains ayant moins de 6 min. de retard  484 non-null    int64  
 7   Minutes de retard                       484 non-null    int64  
 8   Année                                   484 non-null    int64  
dtypes: float64(1), int64(4), object(4)
memory usage: 34.2+ KB


In [33]:
print(df["Instant"].unique())
print(df["Instant.1"].unique())
print(df["Instant.2"].unique())

['Weekends' 'Daluren' 'Avondspits' 'Ochtendspits']
['Weekends' 'Heures creuses' 'Pointe du soir' 'Pointe du matin']
['Weekends' 'Off-peak hours' 'Evening peak hour' 'Morning peak hour']


In [35]:
df_moment = dfs["ponctualite_par_moment"].copy()

df_moment.columns = [
    "date",
    "periode_nl",
    "periode",
    "periode_en",
    "ponctualite_pct",
    "nb_trains",
    "nb_trains_ponctuels",
    "nb_minutes_retard",
    "annee"
]

# Parser la date
df_moment["date"] = pd.to_datetime(df_moment["date"], format="%Y-%m")

# Supprimer les colonnes redondantes
df_moment = df_moment.drop(columns=["periode_nl", "periode_en", "annee"])

print(f"Shape : {df_moment.shape}")
print(f"\nPériodes disponibles : {df_moment['periode'].unique()}")
df_moment.head()

Shape : (484, 6)

Périodes disponibles : ['Weekends' 'Heures creuses' 'Pointe du soir' 'Pointe du matin']


,date,periode,ponctualite_pct,nb_trains,nb_trains_ponctuels,nb_minutes_retard
0,2016-01-01,Weekends,93.869404,24549,23044,32326
1,2016-01-01,Heures creuses,90.689997,45435,41205,93564
2,2016-02-01,Pointe du soir,88.282383,17239,15219,40597
3,2016-03-01,Pointe du matin,88.131496,16548,14584,40055
4,2016-03-01,Pointe du soir,88.470725,17165,15186,41783


### 2.4 — Trains Supprimés

In [36]:
df = dfs["trains_supprimes"]
display(df.head(3))
df.info()

,Mois,Nombre total de trains supprimés,Nombre de trains partiellement supprimés,Nombre de trains entièrement supprimés,Nombre de trains,Pourcentage de trains supprimés,Année
0,2020-10,1920,1405,515,97959,1.922326,2020
1,2020-11,1920,1507,413,88245,2.129429,2020
2,2021-01,2245,1645,600,96572,2.271876,2021


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73 entries, 0 to 72
Data columns (total 7 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Mois                                      73 non-null     object 
 1   Nombre total de trains supprimés          73 non-null     int64  
 2   Nombre de trains partiellement supprimés  73 non-null     int64  
 3   Nombre de trains entièrement supprimés    73 non-null     int64  
 4   Nombre de trains                          73 non-null     int64  
 5   Pourcentage de trains supprimés           73 non-null     float64
 6   Année                                     73 non-null     int64  
dtypes: float64(1), int64(5), object(1)
memory usage: 4.1+ KB


In [37]:
df_suppression = dfs["trains_supprimes"].copy()

df_suppression.columns = [
    "date",
    "nb_trains_supprimes_total",
    "nb_trains_supprimes_partiel",
    "nb_trains_supprimes_entier",
    "nb_trains",
    "pct_trains_supprimes",
    "annee"
]

# Parser la date
df_suppression["date"] = pd.to_datetime(df_suppression["date"], format="%Y-%m")

# Supprimer colonne redondante
df_suppression = df_suppression.drop(columns=["annee"])

print(f"Shape : {df_suppression.shape}")
df_suppression.head()

Shape : (73, 6)


,date,nb_trains_supprimes_total,nb_trains_supprimes_partiel,nb_trains_supprimes_entier,nb_trains,pct_trains_supprimes
0,2020-10-01,1920,1405,515,97959,1.922326
1,2020-11-01,1920,1507,413,88245,2.129429
2,2021-01-01,2245,1645,600,96572,2.271876
3,2021-03-01,2722,2055,667,96579,2.741161
4,2021-05-01,2282,1689,593,94959,2.346747


### 2.5 — KPIs Contrat de Performance

In [38]:
df = dfs["kpi_contrat_performance"]
display(df.head(3))
df.info()

,Année,ID de l'indicateur,Type,Catégorie NL,Catégorie,Subcategorie,Sous-catégorie,Objectif,Valeur,Bonus,Remédiation,Seuil supérieur,Unité
0,2027,S.IP2,Performance,Safety,Safety,Veiligheid van het Infrabel-personeel,Sécurité du personnel d'Infrabel,0.0,NaN,0.0,0.4,NaN,FWI
1,2027,P.IP3​,Performance,Stiptheid,Ponctualité,Vertragingsminuten ten laste van Infrabel - Vr...,Minutes de retard à charge d'Infrabel - Transp...,89000.0,NaN,85000.0,93000.0,NaN,min
2,2027,CSR.IP1,Performance,CSR,CSR,Vermindering CO₂-voetafdruk,Réduction empreinte carbone,58.3,NaN,54.3,62.3,NaN,kton CO₂eq


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 177 entries, 0 to 176
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Année               177 non-null    int64  
 1   ID de l'indicateur  177 non-null    object 
 2   Type                177 non-null    object 
 3   Catégorie NL        177 non-null    object 
 4   Catégorie           177 non-null    object 
 5   Subcategorie        177 non-null    object 
 6   Sous-catégorie      177 non-null    object 
 7   Objectif            73 non-null     float64
 8   Valeur              132 non-null    float64
 9   Bonus               61 non-null     float64
 10  Remédiation         73 non-null     float64
 11  Seuil supérieur     10 non-null     float64
 12  Unité               166 non-null    object 
dtypes: float64(5), int64(1), object(7)
memory usage: 18.1+ KB


In [39]:
df.isna().sum()

Année                   0
ID de l'indicateur      0
Type                    0
Catégorie NL            0
Catégorie               0
Subcategorie            0
Sous-catégorie          0
Objectif              104
Valeur                 45
Bonus                 116
Remédiation           104
Seuil supérieur       167
Unité                  11
dtype: int64

In [42]:
# Quelles catégories existent ?
print("Catégories :")
print(df["Catégorie"].value_counts())

print("\nTypes :")
print(df["Type"].value_counts())

print("\nSous-catégories :")
print(df["Sous-catégorie"].value_counts())

Catégories :
Catégorie
Finance                            38
Ponctualité                        25
Fiabilité & Vitesse Commerciale    20
Safety                             19
Capacité                           19
Chiffres-clés                      18
Asset Management                   16
Relation Client                    10
Accessibilité                       7
CSR                                 5
Name: count, dtype: int64

Types :
Type
Informative    102
Performance     75
Name: count, dtype: int64

Sous-catégories :
Sous-catégorie
Sécurité du personnel d'Infrabel                                 5
Minutes de retard à charge d'Infrabel - Transport voyageurs      5
Minutes de retard à charge d'Infrabel - Transport marchandise    5
Taux de ponctualité commun (avec SNCB)​                          5
Satisfaction clients fret                                        5
                                                                ..
Vitesse commerciale planifiée de trains voyageurs: L     

*
J'ai supprimé les colonnes avec plus de 50% de valeurs manquantes et sans valeur ajoutée pour le dashboard. J'ai conservé objectif et valeur_reelle malgré leurs valeurs manquantes car ce sont les deux colonnes centrales pour comparer performance réelle vs objectif contractuel 


In [43]:
df_kpi = dfs["kpi_contrat_performance"].copy()

# Renommer
df_kpi.columns = [
    "annee", "id_indicateur", "type_indicateur",
    "categorie_nl", "categorie",
    "sous_categorie_nl", "sous_categorie",
    "objectif", "valeur_reelle", "bonus",
    "remediation", "seuil_superieur", "unite"
]

# Garder seulement Ponctualité + Fiabilité
df_kpi = df_kpi[df_kpi["categorie"].isin(["Ponctualité", "Fiabilité & Vitesse Commerciale"])]

# Supprimer colonnes inutiles
df_kpi = df_kpi.drop(columns=[
    "categorie_nl", "sous_categorie_nl",
    "seuil_superieur",
    "bonus", "remediation"
])

print(f"Shape : {df_kpi.shape}")
print(f"\nIndicateurs restants :")
print(df_kpi["sous_categorie"].unique())
df_kpi.head()

Shape : (45, 8)

Indicateurs restants :
["Minutes de retard à charge d'Infrabel - Transport marchandise"
 "Minutes de retard à charge d'Infrabel - Transport voyageurs"
 'Taux de ponctualité commun (avec SNCB)\u200b'
 "Taux de trains voyageurs totalement supprimés à charge d'Infrabel en temps réel"
 "Taux de trains voyageurs partiellement supprimés à charge d'Infrabel en temps réel\u200b"
 'Vitesse commerciale réalisée de trains marchandises par axe\u200b'
 'Vitesse commerciale planifiée de trains voyageurs: L'
 'Vitesse commerciale réalisée de trains voyageurs: L'
 'Vitesse commerciale réalisée de trains voyageurs: P'
 "Taux de ponctualité international voyageurs jusqu'à 6min"
 'Vitesse commerciale réalisée de trains voyageurs: IC'
 'Vitesse commerciale planifiée de trains voyageurs: S'
 'Taux de ponctualité national voyageurs à 6min sur "111" points  (avec la SNCB)'
 "Taux de ponctualité marchandise jusqu'à 30min"
 'Vitesse commerciale planifiée de trains marchandises par axe\u200b'
 

,annee,id_indicateur,type_indicateur,categorie,sous_categorie,objectif,valeur_reelle,unite
1,2027,P.IP3​,Performance,Ponctualité,Minutes de retard à charge d'Infrabel - Transp...,89000.0,NaN,min
8,2027,P.IP2,Performance,Ponctualité,Minutes de retard à charge d'Infrabel - Transp...,362000.0,NaN,min
14,2027,P.IP1​,Performance,Ponctualité,Taux de ponctualité commun (avec SNCB)​,90.6,NaN,%
22,2026,P.IP1​,Performance,Ponctualité,Taux de ponctualité commun (avec SNCB)​,90.5,NaN,%
23,2026,P.IP2,Performance,Ponctualité,Minutes de retard à charge d'Infrabel - Transp...,362000.0,NaN,min


In [47]:
!git add .
!git commit -m "feat: étape 2 — Transform des 5 datasets (nettoyage, renommage, parsing dates)"
!git push 

[main b234a5f] feat: étape 2 — Transform des 5 datasets (nettoyage, renommage, parsing dates)
 1 file changed, 29 insertions(+), 4 deletions(-)
fatal: 'main' does not appear to be a git repository
fatal: Could not read from remote repository.

Please make sure you have the correct access rights
and the repository exists.


---
# ÉTAPE 3 — EDA
> 🔜 À compléter ensemble après le Transform

---
# ÉTAPE 4 — KPIs Métier
> 🔜 À compléter ensemble après l'EDA

---
# ÉTAPE 5 — Visualisation
> 🔜 À compléter ensemble après les KPIs